# LLM-as-a-Judge: The Foundation of Modern LLM Evaluation

This notebook is **the prerequisite** to every other workbook in this series. Before you learn DeepEval, RAGAS, or any evaluation framework, you need to understand the core engine they all run on: **using a powerful LLM to judge the output of another LLM**.

By the end of this notebook you will:

1. Understand why traditional metrics (BLEU, ROUGE, exact match) fail for LLM evaluation
2. Build an LLM-as-a-Judge system from scratch using the OpenAI API
3. Implement the three judge paradigms: pointwise, pairwise, and reference-based
4. Discover and demonstrate four major judge biases
5. See exactly how DeepEval and RAGAS implement LLM-as-a-Judge under the hood
6. Customize the judge: different models, rubrics, temperatures
7. Know when (and when NOT) to use LLM-as-a-Judge

-

## Part 1: What is LLM-as-a-Judge?

### The Problem with Traditional Metrics

For decades, NLP researchers used metrics like:

| Metric | How it works | Why it fails for LLMs |
|---|---|---|
| **BLEU** | N-gram overlap with reference | "Paris is the capital of France" vs "France's capital is Paris" get a low score despite being semantically identical |
| **ROUGE** | Recall-oriented n-gram overlap | Same problem — penalizes paraphrasing |
| **Exact Match** | String equality | Absurdly strict — any rephrasing is a failure |
| **F1 (token)** | Token-level precision/recall | Better, but still ignores meaning |

These metrics compare **strings**, not **meaning**. They were designed for machine translation and summarization where there is a fixed reference answer. But LLM outputs are creative, varied, and context-dependent. A response can be completely correct while sharing zero n-grams with the reference.

### The Core Idea: A Smart Evaluator

What if, instead of counting matching words, we asked a **smart evaluator** that actually understands language?

That is exactly what LLM-as-a-Judge does: use a powerful LLM (typically GPT-4 or GPT-4o) to evaluate the output of another LLM (or the same LLM). The judge reads the question, the response, optionally a reference answer or context, and produces a score with a justification.

**Analogy:** Think of it like having a senior expert review a junior employee's work. The junior writes a report (the LLM response), and the senior (the judge LLM) reads it and gives feedback: "This is 4 out of 5 because the main points are correct but the conclusion is vague."

### The Three Paradigms

LLM-as-a-Judge comes in three flavors:

1. **Pointwise Scoring**: Rate a single response on a scale (e.g., 1-5). "How good is this response?"
2. **Pairwise Comparison**: Given two responses, which one is better? "Is A or B the better answer?"
3. **Reference-Based**: Compare a response to a gold-standard reference. "How close is this to the ideal answer?"

We will build all three from scratch in Part 2.

-

## Part 2: Building LLM-as-a-Judge from Scratch

### Setup

First, let's load our environment and set up the OpenAI client.

In [4]:
import os
import json
import textwrap
from dotenv import load_dotenv

load_dotenv("../.env")

from openai import OpenAI

client = OpenAI()  # picks up OPENAI_API_KEY from environment

# Quick sanity check
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say 'ready' and nothing else."}],
    max_tokens=10,
)
print("OpenAI client ready:", resp.choices[0].message.content)

OpenAI client ready: Ready.


### 2.1 The Simplest Judge: Naive Prompting

The most naive approach: just ask the LLM to rate a response on a 1-5 scale. No structure, no rubric. Let's see what happens.

In [6]:
query = "What causes seasons on Earth?"
response_text = (
    "Seasons are caused by the tilt of Earth's axis (about 23.5 degrees). "
    "As Earth orbits the Sun, different hemispheres receive varying amounts "
    "of direct sunlight, leading to warmer summers and cooler winters."
)

naive_prompt = f"""Rate the following response on a scale of 1 to 5.

Question: {query}
Response: {response_text}

Score:"""

result = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": naive_prompt}],
    temperature=0,
    max_tokens=200,
)

print("Raw judge output:")
print(result.choices[0].message.content)

Raw judge output:
I would rate the response a 5. 

The explanation is clear, concise, and accurately describes the primary cause of seasons on Earth, including the tilt of the axis and its effect on sunlight distribution. It effectively conveys the essential information without unnecessary detail.


**Problems with this approach:**

- The output format is unpredictable: sometimes it returns just "5", sometimes "5/5", sometimes a paragraph
- There is no rubric: what does "3" mean? The LLM decides on its own
- No reasoning: we cannot understand *why* it gave that score
- Difficult to parse programmatically

Let's fix all of these problems.

### 2.2 Better Judge with Structured Output and Rubric

A good LLM judge needs:

1. **A clear rubric** defining what each score means
2. **Structured output** (JSON) so we can parse the result reliably
3. **Mandatory reasoning** so the score is explainable

Let's define a generic quality rubric and build a proper judge prompt.

In [7]:
RUBRIC = """
Score 1 (Terrible): The response is completely wrong, irrelevant, or incoherent.
Score 2 (Poor): The response addresses the topic but contains major errors or is mostly unhelpful.
Score 3 (Adequate): The response is partially correct and somewhat helpful, but has notable gaps or minor errors.
Score 4 (Good): The response is mostly correct, relevant, and helpful with only minor issues.
Score 5 (Excellent): The response is fully correct, comprehensive, clear, and directly addresses the question.
"""

JUDGE_SYSTEM_PROMPT = f"""You are an impartial judge evaluating the quality of an AI assistant's response to a user question.

Use the following rubric:
{RUBRIC}

You MUST respond with valid JSON in this exact format:
{{
  "score": <integer 1-5>,
  "reason": "<one or two sentence justification>"
}}

Do not include any other text outside the JSON."""

print("Judge system prompt defined.")
print("Rubric covers scores 1 (Terrible) through 5 (Excellent).")

Judge system prompt defined.
Rubric covers scores 1 (Terrible) through 5 (Excellent).


In [8]:
def structured_judge(query: str, response_text: str, model: str = "gpt-4o-mini") -> dict:
    """Judge a response using a structured rubric. Returns {score, reason}."""
    user_prompt = f"""Evaluate the following response.

Question: {query}
Response: {response_text}"""

    result = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=300,
    )

    raw = result.choices[0].message.content.strip()
    # Parse JSON from the response
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        # Try to extract JSON from markdown code blocks
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        if match:
            parsed = json.loads(match.group())
        else:
            parsed = {"score": -1, "reason": f"Failed to parse: {raw}"}
    return parsed

print("structured_judge() function defined.")

structured_judge() function defined.


Now let's test the structured judge on several responses of varying quality to see if the rubric produces meaningful, differentiated scores.

In [9]:
query = "What causes seasons on Earth?"

test_responses = {
    "excellent": (
        "Seasons are caused by the tilt of Earth's rotational axis (approximately 23.5 degrees) "
        "relative to its orbital plane around the Sun. As Earth orbits the Sun over the course of "
        "a year, the tilt causes different hemispheres to receive varying amounts of direct sunlight. "
        "When the Northern Hemisphere is tilted toward the Sun, it experiences summer, while the "
        "Southern Hemisphere experiences winter, and vice versa."
    ),
    "adequate": (
        "The seasons happen because of how the Earth moves around the Sun. "
        "Sometimes we are closer and sometimes farther away."
    ),
    "poor": (
        "Seasons are caused by the Sun getting hotter and cooler throughout the year."
    ),
    "irrelevant": (
        "The stock market has been volatile this quarter due to inflation concerns. "
        "Investors should diversify their portfolios."
    ),
}

print(f"Question: {query}\n")
print("=" * 70)

for label, resp in test_responses.items():
    judgment = structured_judge(query, resp)
    print(f"\n[{label.upper()}] Score: {judgment['score']}/5")
    print(f"  Response: {resp[:80]}...")
    print(f"  Reason:   {judgment['reason']}")

Question: What causes seasons on Earth?


[EXCELLENT] Score: 5/5
  Response: Seasons are caused by the tilt of Earth's rotational axis (approximately 23.5 de...
  Reason:   The response is fully correct, comprehensive, and clearly explains the cause of seasons on Earth, addressing the question directly.

[ADEQUATE] Score: 3/5
  Response: The seasons happen because of how the Earth moves around the Sun. Sometimes we a...
  Reason:   The response correctly identifies that the Earth's movement around the Sun causes seasons, but it oversimplifies the explanation by not mentioning the tilt of the Earth's axis, which is a crucial factor in the changing seasons.

[POOR] Score: 2/5
  Response: Seasons are caused by the Sun getting hotter and cooler throughout the year....
  Reason:   The response addresses the topic of seasons but contains a major error by incorrectly attributing the cause to the Sun's temperature changes rather than the tilt of the Earth's axis.

[IRRELEVANT] Score: 1/5
  Res

You should see a clear gradient: the excellent response scores 5, the adequate response around 2-3 (it contains the common misconception about distance rather than tilt), the poor response gets 1-2, and the irrelevant response gets 1. This demonstrates that even a simple structured judge with a rubric can differentiate quality levels.

-

### 2.3 Pointwise Scoring with Criteria

A pointwise judge rates a single response along a specific **criterion** (e.g., relevancy, helpfulness, conciseness). Different criteria can yield very different scores for the same response.

Let's build a flexible pointwise judge that accepts any evaluation criterion.

In [10]:
def pointwise_judge(query: str, response_text: str, criteria: str, model: str = "gpt-4o-mini") -> dict:
    """
    Rate a response on a 1-5 scale for a specific criterion.
    
    Args:
        query: The user's question
        response_text: The LLM's response
        criteria: The evaluation criterion (e.g., 'relevancy', 'helpfulness')
        model: The judge model to use
    
    Returns:
        dict with 'score' (1-5) and 'reason'
    """
    system_prompt = f"""You are an impartial judge. Evaluate the response ONLY on the criterion: {criteria}.

Scoring rubric for {criteria}:
  1 = Completely fails this criterion
  2 = Mostly fails, with minor redeeming aspects
  3 = Partially meets this criterion
  4 = Mostly meets this criterion with minor shortcomings
  5 = Fully meets this criterion

Respond with valid JSON only:
{{"score": <1-5>, "reason": "<brief justification>"}}"""

    user_prompt = f"""Question: {query}
Response: {response_text}

Evaluate on: {criteria}"""

    result = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=300,
    )
    raw = result.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {"score": -1, "reason": f"Parse error: {raw}"}

print("pointwise_judge() function defined.")

pointwise_judge() function defined.


Now let's demonstrate how the **same response** gets different scores depending on which criterion we evaluate. We will use a verbose but accurate response and score it on three criteria: relevancy, helpfulness, and conciseness.

In [11]:
query = "What is the boiling point of water?"

verbose_response = (
    "Great question! Water is a fascinating molecule composed of two hydrogen atoms and one oxygen atom (H2O). "
    "It exists in three states: solid, liquid, and gas. The boiling point of water at standard atmospheric "
    "pressure (1 atm or 101.325 kPa) is 100 degrees Celsius (212 degrees Fahrenheit). This is the temperature "
    "at which water transitions from a liquid to a gaseous state. At higher altitudes, where atmospheric pressure "
    "is lower, water boils at a lower temperature. For example, at the top of Mount Everest, water boils at "
    "approximately 70 degrees Celsius. The boiling point is also affected by dissolved substances; adding salt "
    "to water raises its boiling point slightly, a phenomenon known as boiling point elevation. This principle "
    "is widely used in cooking and industrial processes."
)

criteria_list = ["relevancy", "helpfulness", "conciseness"]

print(f"Question: {query}")
print(f"Response: {verbose_response[:100]}...\n")
print("=" * 60)

for criterion in criteria_list:
    result = pointwise_judge(query, verbose_response, criterion)
    print(f"\n  {criterion.upper()}: {result['score']}/5")
    print(f"  Reason: {result['reason']}")

Question: What is the boiling point of water?
Response: Great question! Water is a fascinating molecule composed of two hydrogen atoms and one oxygen atom (...


  RELEVANCY: 5/5
  Reason: The response directly answers the question about the boiling point of water and provides relevant additional information about its properties and factors affecting the boiling point.

  HELPFULNESS: 5/5
  Reason: The response provides a clear and accurate answer to the question about the boiling point of water, along with additional relevant information about the effects of altitude and dissolved substances, enhancing the overall helpfulness.

  CONCISENESS: 2/5
  Reason: The response provides excessive detail and background information that is not directly relevant to the question about the boiling point of water, making it less concise.


Notice how the same response likely scores high on relevancy and helpfulness (it correctly answers the question and provides useful context) but lower on conciseness (it includes a lot of extra information). This is the power of criterion-specific evaluation: it lets you measure exactly the dimension of quality you care about.

-

### 2.4 Pairwise Comparison

Instead of scoring a single response, a pairwise judge compares two responses and decides which is better. This is often more reliable than pointwise scoring because relative comparisons are easier for both humans and LLMs than absolute ratings.

However, pairwise comparison has a well-known flaw: **position bias** (the judge tends to prefer whichever response is listed first). We will demonstrate this and implement a debiasing technique.

In [12]:
def pairwise_judge(query: str, response_a: str, response_b: str, model: str = "gpt-4o-mini") -> dict:
    """
    Compare two responses and determine which is better.
    
    Returns:
        dict with 'winner' ('A' or 'B' or 'tie') and 'reason'
    """
    system_prompt = """You are an impartial judge comparing two AI responses to a user question.
Evaluate both responses on overall quality: correctness, relevance, helpfulness, and clarity.

Respond with valid JSON only:
{"winner": "A" or "B" or "tie", "reason": "<brief justification>"}"""

    user_prompt = f"""Question: {query}

Response A: {response_a}

Response B: {response_b}

Which response is better?"""

    result = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=300,
    )
    raw = result.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {"winner": "error", "reason": f"Parse error: {raw}"}

print("pairwise_judge() function defined.")

pairwise_judge() function defined.


Let's test with a clear winner case first.

In [13]:
query = "What is the capital of Japan?"

good_response = "The capital of Japan is Tokyo. It has been the capital since 1868 when the Emperor moved there from Kyoto."
bad_response = "The capital of Japan is Osaka. It is known for its street food."

result = pairwise_judge(query, good_response, bad_response)
print(f"Winner: Response {result['winner']}")
print(f"Reason: {result['reason']}")

Winner: Response A
Reason: Response A is correct and provides relevant historical context, while Response B contains incorrect information.


Now let's demonstrate **position bias**. We will use two responses of similar quality and swap their positions to see if the judge's preference changes.

In [14]:
query = "Explain the concept of supply and demand in economics."

response_1 = (
    "Supply and demand is a fundamental economic model. Demand refers to how much of a product "
    "consumers want to buy at various prices - generally, lower prices lead to higher demand. "
    "Supply refers to how much producers are willing to sell at various prices - higher prices "
    "incentivize more supply. The equilibrium price is where supply meets demand."
)

response_2 = (
    "In economics, supply and demand describes the relationship between sellers and buyers in a market. "
    "The demand curve shows that people buy more when prices are low. The supply curve shows that "
    "producers make more when prices are high. Where these curves intersect is the market equilibrium, "
    "which determines the price and quantity of goods traded."
)

# Test 1: response_1 as A, response_2 as B
result_ab = pairwise_judge(query, response_1, response_2)
print("Order 1: response_1=A, response_2=B")
print(f"  Winner: {result_ab['winner']}")
print(f"  Reason: {result_ab['reason']}")

# Test 2: Swap positions
result_ba = pairwise_judge(query, response_2, response_1)
print(f"\nOrder 2: response_2=A, response_1=B")
print(f"  Winner: {result_ba['winner']}")
print(f"  Reason: {result_ba['reason']}")

# Analysis
print("\n" + "=" * 60)
if result_ab['winner'] == 'A' and result_ba['winner'] == 'A':
    print("POSITION BIAS DETECTED: The judge always preferred whichever response was listed as 'A'.")
elif result_ab['winner'] == result_ba['winner']:
    print(f"Consistent: The judge preferred the same response in both orderings.")
else:
    winner_1 = "response_1" if result_ab['winner'] == 'A' else "response_2"
    winner_2 = "response_2" if result_ba['winner'] == 'A' else "response_1"
    print(f"Order 1 preferred: {winner_1}")
    print(f"Order 2 preferred: {winner_2}")
    if winner_1 == winner_2:
        print("Consistent result despite position swap.")
    else:
        print("INCONSISTENCY detected - results changed when positions were swapped.")

Order 1: response_1=A, response_2=B
  Winner: A
  Reason: Response A provides a clear and concise explanation of supply and demand, covering key concepts like demand, supply, and equilibrium price effectively. While Response B is also relevant, it is slightly less clear and more technical in its language.

Order 2: response_2=A, response_1=B
  Winner: tie
  Reason: Both responses accurately explain the concept of supply and demand, covering key elements such as the relationship between price and quantity for both demand and supply, as well as the concept of market equilibrium. They are both clear and relevant.

Order 1 preferred: response_1
Order 2 preferred: response_1
Consistent result despite position swap.


### Debiasing Pairwise Comparison

The standard mitigation for position bias is to run the comparison **twice** (swapping the order) and take the consensus. If the two runs disagree, it is declared a tie. This is exactly what many evaluation frameworks (including Chatbot Arena) do.

In [15]:
def debiased_pairwise_judge(query: str, response_a: str, response_b: str, model: str = "gpt-4o-mini") -> dict:
    """
    Run pairwise comparison in both orderings and take consensus.
    If the two orderings disagree, declare a tie.
    """
    # Run 1: A first, B second
    result_1 = pairwise_judge(query, response_a, response_b, model)
    # Map the winner to the original labels
    winner_1 = result_1['winner']  # 'A', 'B', or 'tie'
    
    # Run 2: B first, A second (swapped)
    result_2 = pairwise_judge(query, response_b, response_a, model)
    # Map back: if winner is 'A' in run 2, that means response_b won (since B was listed as A)
    winner_2_mapped = {'A': 'B', 'B': 'A', 'tie': 'tie'}.get(result_2['winner'], 'tie')
    
    # Consensus
    if winner_1 == winner_2_mapped:
        final_winner = winner_1
        consensus = "agree"
    else:
        final_winner = "tie"
        consensus = "disagree"
    
    return {
        "winner": final_winner,
        "consensus": consensus,
        "run1": {"order": "A=original_A, B=original_B", "winner": winner_1, "reason": result_1['reason']},
        "run2": {"order": "A=original_B, B=original_A", "winner": result_2['winner'], "mapped_winner": winner_2_mapped, "reason": result_2['reason']},
    }

# Test debiased comparison
result = debiased_pairwise_judge(query, response_1, response_2)

print("Debiased Pairwise Comparison")
print("=" * 60)
print(f"Final Winner: {result['winner']} (consensus: {result['consensus']})")
print(f"\nRun 1 ({result['run1']['order']}):")
print(f"  Winner: {result['run1']['winner']} - {result['run1']['reason']}")
print(f"\nRun 2 ({result['run2']['order']}):")
print(f"  Raw winner: {result['run2']['winner']}, Mapped: {result['run2']['mapped_winner']}")
print(f"  Reason: {result['run2']['reason']}")

Debiased Pairwise Comparison
Final Winner: tie (consensus: disagree)

Run 1 (A=original_A, B=original_B):
  Winner: A - Response A provides a clear and concise explanation of supply and demand, covering key concepts like demand, supply, and equilibrium price in a straightforward manner.

Run 2 (A=original_B, B=original_A):
  Raw winner: tie, Mapped: tie
  Reason: Both responses accurately explain the concept of supply and demand, covering key elements such as the relationship between price and quantity for both demand and supply, as well as the concept of market equilibrium. They are both clear and relevant.


-

### 2.5 Reference-Based Scoring

The third paradigm compares a candidate response against a **gold-standard reference answer**. This is useful when you have human-written ideal answers. The judge evaluates how closely the candidate captures the same information as the reference.

In [16]:
def reference_judge(query: str, response_text: str, reference: str, model: str = "gpt-4o-mini") -> dict:
    """
    Score how well a response matches a reference answer.
    
    Returns:
        dict with 'score' (1-5) and 'reason'
    """
    system_prompt = """You are an impartial judge. Compare the candidate response to the reference answer.

Scoring rubric:
  1 = Completely different or contradicts the reference
  2 = Captures very little of the reference information
  3 = Captures some key points but misses important details
  4 = Captures most key points with minor omissions or additions
  5 = Captures all key points from the reference (paraphrasing is fine)

Respond with valid JSON only:
{"score": <1-5>, "reason": "<brief justification>"}"""

    user_prompt = f"""Question: {query}

Reference Answer: {reference}

Candidate Response: {response_text}"""

    result = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0,
        max_tokens=300,
    )
    raw = result.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
        return {"score": -1, "reason": f"Parse error: {raw}"}

print("reference_judge() function defined.")

reference_judge() function defined.


Let's test the reference-based judge with four candidate responses of varying similarity to the reference.

In [17]:
query = "What is photosynthesis?"
reference = (
    "Photosynthesis is the biological process by which green plants and certain other organisms "
    "convert light energy (usually from the Sun) into chemical energy stored in glucose. It occurs "
    "primarily in the chloroplasts of plant cells, using carbon dioxide from the air and water from "
    "the soil. Oxygen is released as a byproduct."
)

candidates = {
    "exact_paraphrase": (
        "Photosynthesis is a process where plants use sunlight to turn CO2 and water into glucose "
        "and oxygen. This happens in the chloroplasts of their cells."
    ),
    "partial_match": (
        "Photosynthesis is how plants make food using sunlight. It involves the green parts of the plant."
    ),
    "wrong_but_related": (
        "Photosynthesis is the process where plants absorb oxygen and release carbon dioxide "
        "to produce energy for growth."
    ),
    "completely_wrong": (
        "Photosynthesis is a type of nuclear fusion that occurs in the Earth's core, "
        "generating heat that sustains life on the surface."
    ),
}

print(f"Question: {query}")
print(f"Reference: {reference[:80]}...\n")
print("=" * 70)

for label, candidate in candidates.items():
    result = reference_judge(query, candidate, reference)
    print(f"\n[{label.upper()}] Score: {result['score']}/5")
    print(f"  Candidate: {candidate[:80]}...")
    print(f"  Reason:    {result['reason']}")

Question: What is photosynthesis?
Reference: Photosynthesis is the biological process by which green plants and certain other...


[EXACT_PARAPHRASE] Score: 5/5
  Candidate: Photosynthesis is a process where plants use sunlight to turn CO2 and water into...
  Reason:    The candidate response captures all key points from the reference answer, including the conversion of light energy into chemical energy, the role of chloroplasts, and the use of carbon dioxide and water, as well as the production of oxygen.

[PARTIAL_MATCH] Score: 2/5
  Candidate: Photosynthesis is how plants make food using sunlight. It involves the green par...
  Reason:    The candidate response captures the basic idea of photosynthesis but lacks key details such as the conversion of light energy into chemical energy, the role of chloroplasts, the use of carbon dioxide and water, and the release of oxygen as a byproduct.

[WRONG_BUT_RELATED] Score: 1/5
  Candidate: Photosynthesis is the process where plants absorb ox

The reference-based judge should clearly separate the paraphrase (high score) from the wrong answer (low score), with the partial match and the reversed-facts response falling in between. This paradigm is especially useful for RAG evaluation where you have ground-truth answers.

-

## Part 3: Judge Biases -- Demonstrated with Code

LLM judges are not perfect. They exhibit systematic biases that can skew evaluation results. Understanding these biases is critical before relying on LLM-as-a-Judge in production.

We will demonstrate four major biases:

1. **Position Bias** (already shown above)
2. **Verbosity Bias**: longer responses score higher, regardless of quality
3. **Self-Enhancement Bias**: the judge rates its own outputs higher
4. **Format Bias**: well-formatted responses (markdown, JSON) score higher than plain text

### 3.1 Verbosity Bias

LLM judges tend to prefer verbose responses even when a concise response is equally or more accurate. Let's demonstrate this by creating a short, accurate response and a verbose, slightly less accurate response.

In [18]:
query = "What is the speed of light?"

concise_response = "The speed of light in a vacuum is approximately 299,792,458 meters per second (about 3 x 10^8 m/s)."

verbose_response = (
    "That's a great question! The speed of light is one of the most fundamental constants in physics. "
    "Light travels at different speeds depending on the medium it passes through. In a vacuum, which is "
    "the absence of matter, light travels at its maximum speed. This speed was first measured with "
    "reasonable accuracy by Ole Roemer in 1676 using observations of Jupiter's moon Io. Over the centuries, "
    "scientists have refined this measurement using increasingly sophisticated equipment. Today, the speed "
    "of light in a vacuum is defined as exactly 299,792,458 meters per second. This value is used to "
    "define the meter itself in the International System of Units. Albert Einstein's theory of special "
    "relativity, published in 1905, showed that the speed of light is the cosmic speed limit - nothing "
    "with mass can reach or exceed this speed. The speed of light is commonly denoted by the letter 'c' "
    "in physics equations, most famously in E=mc^2."
)

# Score both
concise_score = structured_judge(query, concise_response)
verbose_score = structured_judge(query, verbose_response)

print("VERBOSITY BIAS DEMONSTRATION")
print("=" * 60)
print(f"\nConcise response ({len(concise_response)} chars): Score {concise_score['score']}/5")
print(f"  Reason: {concise_score['reason']}")
print(f"\nVerbose response ({len(verbose_response)} chars): Score {verbose_score['score']}/5")
print(f"  Reason: {verbose_score['reason']}")

if verbose_score['score'] > concise_score['score']:
    print("\n>>> VERBOSITY BIAS: The longer response scored higher despite both being correct.")
elif verbose_score['score'] == concise_score['score']:
    print("\n>>> No verbosity bias detected here (both scored equally).")
else:
    print("\n>>> Concise response actually scored higher -- no verbosity bias.")

VERBOSITY BIAS DEMONSTRATION

Concise response (99 chars): Score 5/5
  Reason: The response is fully correct, comprehensive, and directly addresses the question about the speed of light in a vacuum with precise information.

Verbose response (932 chars): Score 5/5
  Reason: The response is fully correct, comprehensive, and clearly addresses the question about the speed of light, providing historical context and its significance in physics.

>>> No verbosity bias detected here (both scored equally).


### Mitigating Verbosity Bias

We can mitigate verbosity bias by explicitly instructing the judge that **brevity should not be penalized** and that **unnecessary verbosity should not be rewarded**.

In [19]:
DEBIASED_JUDGE_PROMPT = """You are an impartial judge evaluating the quality of an AI response.

IMPORTANT: Do NOT favor longer responses. A concise, accurate response is just as good as
a verbose one. Evaluate based on correctness and relevance, NOT length. Unnecessary verbosity
should be considered a slight negative, not a positive.

Rubric:
  1 = Completely wrong or irrelevant
  2 = Mostly wrong with minor correct elements
  3 = Partially correct but has notable gaps or errors
  4 = Mostly correct and relevant with minor issues
  5 = Fully correct, relevant, and appropriately detailed

Respond with valid JSON: {"score": <1-5>, "reason": "<justification>"}"""

def debiased_judge(query: str, response_text: str, model: str = "gpt-4o-mini") -> dict:
    result = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": DEBIASED_JUDGE_PROMPT},
            {"role": "user", "content": f"Question: {query}\nResponse: {response_text}"},
        ],
        temperature=0,
        max_tokens=300,
    )
    raw = result.choices[0].message.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        return json.loads(match.group()) if match else {"score": -1, "reason": raw}

# Re-test with debiased judge
concise_debiased = debiased_judge(query, concise_response)
verbose_debiased = debiased_judge(query, verbose_response)

print("DEBIASED JUDGE RESULTS")
print("=" * 60)
print(f"Concise: {concise_debiased['score']}/5 - {concise_debiased['reason']}")
print(f"Verbose: {verbose_debiased['score']}/5 - {verbose_debiased['reason']}")

DEBIASED JUDGE RESULTS
Concise: 5/5 - The response is fully correct, relevant, and provides the precise value of the speed of light in a vacuum, along with a useful approximation.
Verbose: 5/5 - The response accurately defines the speed of light, provides its value in a vacuum, explains its significance in physics, and mentions historical context and its role in special relativity. All information is relevant and correct.


### 3.2 Self-Enhancement Bias

When the same model that generated a response also judges it, it tends to rate its own output higher than a response written by a different source. Let's demonstrate this.

In [ ]:
query = "Explain the difference between machine learning and deep learning."

# Generate a response using GPT-4o-mini
gen_result = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": query}],
    temperature=0.7,
    max_tokens=300,
)
llm_generated = gen_result.choices[0].message.content

# Human-written response (roughly equivalent quality)
human_written = (
    "Machine learning is a broad field of AI where algorithms learn patterns from data to make "
    "predictions or decisions. It includes techniques like linear regression, decision trees, and SVMs. "
    "Deep learning is a subset of machine learning that uses artificial neural networks with multiple "
    "layers (hence 'deep'). Deep learning excels at tasks involving unstructured data like images, audio, "
    "and text, but requires significantly more data and computational resources than traditional ML."
)

print("LLM-generated response:")
print(textwrap.fill(llm_generated[:300], width=80))
print("\nHuman-written response:")
print(textwrap.fill(human_written, width=80))

In [ ]:
# Judge both with GPT-4o-mini (same model that generated one of them)
llm_score = structured_judge(query, llm_generated, model="gpt-4o-mini")
human_score = structured_judge(query, human_written, model="gpt-4o-mini")

print("SELF-ENHANCEMENT BIAS TEST")
print("Judge model: gpt-4o-mini (same model that generated one response)")
print("=" * 60)
print(f"\nLLM-generated (by gpt-4o-mini): {llm_score['score']}/5")
print(f"  Reason: {llm_score['reason']}")
print(f"\nHuman-written:                  {human_score['score']}/5")
print(f"  Reason: {human_score['reason']}")

if llm_score['score'] > human_score['score']:
    print("\n>>> SELF-ENHANCEMENT BIAS: The model rated its own output higher.")
elif llm_score['score'] == human_score['score']:
    print("\n>>> Both scored equally -- bias not clearly evident in this run.")
else:
    print("\n>>> Human response rated higher -- no self-enhancement bias detected.")

**Note**: Self-enhancement bias may not always manifest in a single comparison. Research (Zheng et al., 2023) shows it is a statistical tendency observed across many comparisons. The mitigation is to use a **different (preferably stronger) model** as the judge.

-

### 3.3 Format Bias

LLM judges tend to score well-formatted responses (using markdown headers, bullet points, bold text) higher than plain-text responses with identical content.

In [ ]:
query = "What are the benefits of exercise?"

plain_text = (
    "Exercise has many benefits. It improves cardiovascular health by strengthening the heart. "
    "It helps with weight management by burning calories. It boosts mental health by releasing "
    "endorphins. It strengthens muscles and bones. It also improves sleep quality."
)

formatted_text = (
    "## Benefits of Exercise\n\n"
    "Exercise has many benefits:\n\n"
    "- **Cardiovascular Health**: Strengthens the heart\n"
    "- **Weight Management**: Burns calories effectively\n"
    "- **Mental Health**: Releases endorphins that boost mood\n"
    "- **Musculoskeletal**: Strengthens muscles and bones\n"
    "- **Sleep Quality**: Improves overall sleep patterns"
)

plain_score = structured_judge(query, plain_text)
formatted_score = structured_judge(query, formatted_text)

print("FORMAT BIAS DEMONSTRATION")
print("(Same content, different formatting)")
print("=" * 60)
print(f"\nPlain text:    {plain_score['score']}/5 - {plain_score['reason']}")
print(f"Formatted:     {formatted_score['score']}/5 - {formatted_score['reason']}")

if formatted_score['score'] > plain_score['score']:
    print("\n>>> FORMAT BIAS: The formatted response scored higher despite identical content.")
elif formatted_score['score'] == plain_score['score']:
    print("\n>>> No format bias detected (both scored equally).")
else:
    print("\n>>> Plain text scored higher -- no format bias.")

### 3.4 Summary of Biases

| Bias | Description | Mitigation |
|---|---|---|
| **Position Bias** | First-listed response preferred in pairwise comparison | Run both orderings, take consensus |
| **Verbosity Bias** | Longer responses score higher | Add "do not favor length" to rubric |
| **Self-Enhancement** | Model rates its own outputs higher | Use a different/stronger model as judge |
| **Format Bias** | Markdown/structured responses score higher | Add "ignore formatting" to rubric, or strip formatting before judging |

These biases are well-documented in the literature (Zheng et al., 2023 "Judging LLM-as-a-Judge") and should always be considered when designing evaluation pipelines.

-

## Part 4: How DeepEval Uses LLM-as-a-Judge Under the Hood

Every DeepEval LLM metric is, at its core, an LLM-as-a-Judge pipeline. The framework structures the prompts, manages the LLM calls, and computes scores in a standardized way. Let's peek behind the curtain and see exactly what happens when you call `metric.measure()`.

### 4.1 Faithfulness: The Manual Way

The Faithfulness metric measures whether the LLM response is supported by the provided context (i.e., the response does not hallucinate). Here is DeepEval's approach, step by step:

1. **Extract claims** from the response (LLM call #1)
2. **Verify each claim** against the context (LLM call #2)
3. **Score** = number of verified claims / total claims

Let's simulate this manually with the OpenAI API.

In [20]:
# Our test case for faithfulness
test_query = "What is the history of the Eiffel Tower?"

test_context = [
    "The Eiffel Tower was built from 1887 to 1889 as the centerpiece of the 1889 World's Fair in Paris.",
    "It was designed by Gustave Eiffel's engineering company.",
    "The tower stands 330 meters tall and was the world's tallest man-made structure until 1930.",
    "Initially criticized by some of France's leading artists and intellectuals, the tower has become a global cultural icon of France.",
]

# Response with some faithful claims and one hallucinated claim
test_response = (
    "The Eiffel Tower was built between 1887 and 1889 for the World's Fair in Paris. "
    "It was designed by Gustave Eiffel's company and stands 330 meters tall. "
    "The tower was the tallest structure in the world until 1930. "
    "It cost approximately 7.8 million francs to build and attracted over 2 million visitors in its first year."
)

print("Test case defined.")
print(f"Query:    {test_query}")
print(f"Context:  {len(test_context)} passages")
print(f"Response: {test_response[:80]}...")

Test case defined.
Query:    What is the history of the Eiffel Tower?
Context:  4 passages
Response: The Eiffel Tower was built between 1887 and 1889 for the World's Fair in Paris. ...


**Step 1: Extract claims from the response.** We ask the LLM to break the response into individual factual claims.

In [21]:
# Step 1: Extract claims
extract_prompt = f"""Extract all factual claims from the following response. 
Return a JSON array of strings, where each string is one distinct factual claim.

Response: {test_response}

Return ONLY a JSON array, nothing else."""

claims_result = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": extract_prompt}],
    temperature=0,
    max_tokens=500,
)

claims_raw = claims_result.choices[0].message.content.strip()
# Clean up possible markdown code blocks
if claims_raw.startswith("```"):
    claims_raw = claims_raw.split("```")[1]
    if claims_raw.startswith("json"):
        claims_raw = claims_raw[4:]
claims = json.loads(claims_raw.strip())

print(f"Extracted {len(claims)} claims:")
for i, claim in enumerate(claims, 1):
    print(f"  {i}. {claim}")

Extracted 6 claims:
  1. The Eiffel Tower was built between 1887 and 1889 for the World's Fair in Paris.
  2. It was designed by Gustave Eiffel's company.
  3. The Eiffel Tower stands 330 meters tall.
  4. The Eiffel Tower was the tallest structure in the world until 1930.
  5. It cost approximately 7.8 million francs to build.
  6. The Eiffel Tower attracted over 2 million visitors in its first year.


**Step 2: Verify each claim against the context.** For each claim, we ask the LLM whether it can be supported by the provided context passages.

In [22]:
# Step 2: Verify each claim
context_text = "\n".join(test_context)

verdicts = []
for claim in claims:
    verify_prompt = f"""Determine if the following claim is supported by the given context.

Context:
{context_text}

Claim: {claim}

Respond with JSON: {{"verdict": "supported" or "not_supported", "reason": "<brief explanation>"}}"""

    verify_result = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": verify_prompt}],
        temperature=0,
        max_tokens=200,
    )
    raw = verify_result.choices[0].message.content.strip()
    try:
        verdict = json.loads(raw)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        verdict = json.loads(match.group()) if match else {"verdict": "error", "reason": raw}
    verdicts.append({"claim": claim, **verdict})

print("Claim Verification Results:")
print("=" * 70)
for v in verdicts:
    status = "SUPPORTED" if v['verdict'] == 'supported' else "NOT SUPPORTED"
    print(f"  [{status}] {v['claim']}")
    print(f"           Reason: {v['reason']}")

Claim Verification Results:
  [SUPPORTED] The Eiffel Tower was built between 1887 and 1889 for the World's Fair in Paris.
           Reason: The context states that the Eiffel Tower was built from 1887 to 1889 as the centerpiece of the 1889 World's Fair in Paris, which directly supports the claim.
  [SUPPORTED] It was designed by Gustave Eiffel's company.
           Reason: The context explicitly states that the Eiffel Tower was designed by Gustave Eiffel's engineering company.
  [SUPPORTED] The Eiffel Tower stands 330 meters tall.
           Reason: The context explicitly states that the Eiffel Tower stands 330 meters tall.
  [SUPPORTED] The Eiffel Tower was the tallest structure in the world until 1930.
           Reason: The context states that the Eiffel Tower was the world's tallest man-made structure until 1930, which directly supports the claim.
  [NOT SUPPORTED] It cost approximately 7.8 million francs to build.
           Reason: The context does not provide any information ab

**Step 3: Calculate the faithfulness score.**

In [23]:
# Step 3: Calculate score
supported_count = sum(1 for v in verdicts if v['verdict'] == 'supported')
total_count = len(verdicts)
manual_faithfulness_score = supported_count / total_count if total_count > 0 else 0

print(f"Supported claims: {supported_count}/{total_count}")
print(f"Manual Faithfulness Score: {manual_faithfulness_score:.2f}")
print(f"\nThis means {manual_faithfulness_score*100:.0f}% of the response's claims are grounded in the provided context.")

Supported claims: 4/6
Manual Faithfulness Score: 0.67

This means 67% of the response's claims are grounded in the provided context.


Now let's run the exact same test case through DeepEval's `FaithfulnessMetric` and compare.

In [26]:
from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase

faithfulness_metric = FaithfulnessMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    verbose_mode=True,
)

deepeval_test_case = LLMTestCase(
    input=test_query,
    actual_output=test_response,
    retrieval_context=test_context,
)

faithfulness_metric.measure(deepeval_test_case)

print(f"\nDeepEval Faithfulness Score: {faithfulness_metric.score:.2f}")
print(f"Reason: {faithfulness_metric.reason}")
print(f"\nManual Score:   {manual_faithfulness_score:.2f}")
print(f"DeepEval Score: {faithfulness_metric.score:.2f}")
print(f"\nThe scores may differ slightly because DeepEval uses more sophisticated prompts "
      f"and may extract/verify claims differently, but the underlying approach is the same.")

Output()

**************************************************

Faithfulness Verbose Logs

**************************************************

Truths (limit=None):
[
    "The Eiffel Tower was built from 1887 to 1889.",
    "The Eiffel Tower was the centerpiece of the 1889 World's Fair in Paris.",
    "The Eiffel Tower was designed by Gustave Eiffel's engineering company.",
    "The Eiffel Tower stands 330 meters tall.",
    "The Eiffel Tower was the world's tallest man-made structure until 1930.",
    "The Eiffel Tower was initially criticized by some of France's leading artists and intellectuals.",
    "The Eiffel Tower has become a global cultural icon of France."
] 
 
Claims:
[
    "The Eiffel Tower was built between 1887 and 1889 for the World's Fair in Paris.",
    "The Eiffel Tower was designed by Gustave Eiffel's company.",
    "The Eiffel Tower stands 330 meters tall.",
    "The Eiffel Tower was the tallest structure in the world until 1930.",
    "It cost approximately 7.8 million francs to build the Eiffel Tower.",
    "The Eiffel Tower attracted over 2 million visitors in its first year."
] 
 
Verdicts:
[
    {
        "verdict": "no",
        "reason": "The Eiffel Tower was built from 1887 to 1889, but it was specifically built for the 1889 World's
Fair, not 'between 1887 and 1889 for the World's Fair'."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "idk",
        "reason": "The cost of building the Eiffel Tower is not mentioned in the retrieval context."
    },
    {
        "verdict": "idk",
        "reason": "The number of visitors in the first year is not mentioned in the retrieval context."
    }
]
 
Score: 0.8333333333333334
Reason: The score is 0.83 because the actual output inaccurately states that the Eiffel Tower was built 'between 
1887 and 1889 for the World's Fair', while it was specifically built for the 1889 World's Fair, which creates a 
slight misalignment with the retrieval context.

======================================================================


DeepEval Faithfulness Score: 0.83
Reason: The score is 0.83 because the actual output inaccurately states that the Eiffel Tower was built 'between 1887 and 1889 for the World's Fair', while it was specifically built for the 1889 World's Fair, which creates a slight misalignment with the retrieval context.

Manual Score:   0.67
DeepEval Score: 0.83

The scores may differ slightly because DeepEval uses more sophisticated prompts and may extract/verify claims differently, but the underlying approach is the same.


### 4.2 Answer Relevancy: The Manual Way

The Answer Relevancy metric measures whether the response is relevant to the question asked. DeepEval's approach:

1. **Break the response into statements** (LLM call #1)
2. **Classify each statement** as relevant or irrelevant to the query (LLM call #2)
3. **Score** = relevant statements / total statements

Let's simulate this.

In [27]:
relevancy_query = "What are the health benefits of green tea?"

relevancy_response = (
    "Green tea is rich in antioxidants called catechins, which can help reduce cell damage. "
    "Studies suggest it may lower the risk of heart disease and improve cholesterol levels. "
    "Green tea also contains L-theanine, an amino acid that can improve brain function. "
    "By the way, coffee is also a popular beverage worldwide with its own set of benefits. "
    "Green tea may also help with weight loss by boosting metabolism."
)

# Step 1: Extract statements
extract_stmt_prompt = f"""Break the following response into individual statements.
Return a JSON array of strings.

Response: {relevancy_response}

Return ONLY a JSON array."""

stmt_result = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": extract_stmt_prompt}],
    temperature=0,
    max_tokens=500,
)

stmt_raw = stmt_result.choices[0].message.content.strip()
if stmt_raw.startswith("```"):
    stmt_raw = stmt_raw.split("```")[1]
    if stmt_raw.startswith("json"):
        stmt_raw = stmt_raw[4:]
statements = json.loads(stmt_raw.strip())

print(f"Extracted {len(statements)} statements:")
for i, s in enumerate(statements, 1):
    print(f"  {i}. {s}")

Extracted 5 statements:
  1. Green tea is rich in antioxidants called catechins, which can help reduce cell damage.
  2. Studies suggest it may lower the risk of heart disease and improve cholesterol levels.
  3. Green tea also contains L-theanine, an amino acid that can improve brain function.
  4. By the way, coffee is also a popular beverage worldwide with its own set of benefits.
  5. Green tea may also help with weight loss by boosting metabolism.


In [28]:
# Step 2: Classify each statement as relevant or irrelevant
relevancy_verdicts = []
for stmt in statements:
    classify_prompt = f"""Is the following statement relevant to answering the question?

Question: {relevancy_query}
Statement: {stmt}

Respond with JSON: {{"relevant": true or false, "reason": "<brief explanation>"}}"""

    classify_result = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": classify_prompt}],
        temperature=0,
        max_tokens=200,
    )
    raw = classify_result.choices[0].message.content.strip()
    try:
        verdict = json.loads(raw)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        verdict = json.loads(match.group()) if match else {"relevant": False, "reason": raw}
    relevancy_verdicts.append({"statement": stmt, **verdict})

print("Statement Relevancy Classification:")
print("=" * 70)
for v in relevancy_verdicts:
    status = "RELEVANT" if v['relevant'] else "IRRELEVANT"
    print(f"  [{status}] {v['statement']}")
    print(f"           {v['reason']}")

Statement Relevancy Classification:
  [RELEVANT] Green tea is rich in antioxidants called catechins, which can help reduce cell damage.
           The statement provides specific information about the antioxidants in green tea and their potential health benefits, directly addressing the question about the health benefits of green tea.
  [RELEVANT] Studies suggest it may lower the risk of heart disease and improve cholesterol levels.
           The statement directly addresses the health benefits of green tea by mentioning its potential effects on heart disease and cholesterol levels.
  [RELEVANT] Green tea also contains L-theanine, an amino acid that can improve brain function.
           The statement provides information about a component of green tea (L-theanine) that contributes to its health benefits, specifically in relation to brain function.
  [IRRELEVANT] By the way, coffee is also a popular beverage worldwide with its own set of benefits.
           The statement discusses co

In [29]:
# Step 3: Calculate relevancy score
relevant_count = sum(1 for v in relevancy_verdicts if v['relevant'])
total_stmts = len(relevancy_verdicts)
manual_relevancy_score = relevant_count / total_stmts if total_stmts > 0 else 0

print(f"Relevant statements: {relevant_count}/{total_stmts}")
print(f"Manual Answer Relevancy Score: {manual_relevancy_score:.2f}")
print(f"\nThe statement about coffee should be classified as irrelevant, reducing the score slightly.")

Relevant statements: 4/5
Manual Answer Relevancy Score: 0.80

The statement about coffee should be classified as irrelevant, reducing the score slightly.


Let's verify with DeepEval.

In [30]:
from deepeval.metrics import AnswerRelevancyMetric

ar_metric = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini",
)

ar_test_case = LLMTestCase(
    input=relevancy_query,
    actual_output=relevancy_response,
)

ar_metric.measure(ar_test_case)

print(f"DeepEval Answer Relevancy: {ar_metric.score:.2f}")
print(f"Reason: {ar_metric.reason}")
print(f"\nManual Score:   {manual_relevancy_score:.2f}")
print(f"DeepEval Score: {ar_metric.score:.2f}")

Output()

DeepEval Answer Relevancy: 0.80
Reason: The score is 0.80 because while the response provided valuable information about green tea's health benefits, it included irrelevant statements about coffee that detracted from the focus on green tea.

Manual Score:   0.80
DeepEval Score: 0.80


### 4.3 G-Eval: Chain-of-Thought Evaluation

G-Eval (Liu et al., 2023) is a more sophisticated LLM-as-a-Judge approach that works in two phases:

1. **Generate Chain-of-Thought (CoT) evaluation steps** from the criteria description
2. **Use the CoT steps to evaluate** the response, producing a finer-grained score

DeepEval implements G-Eval as the `GEval` metric, allowing you to define completely custom evaluation criteria.

Let's use G-Eval to evaluate "helpfulness" — a criterion that is not a built-in metric.

In [31]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

helpfulness_metric = GEval(
    name="Helpfulness",
    criteria=(
        "Helpfulness - determine if the response provides practical, actionable information "
        "that would genuinely help the user accomplish their goal. The response should not just "
        "be correct, but should be useful and applicable."
    ),
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

# Test on a helpful response
helpful_case = LLMTestCase(
    input="How can I improve my sleep quality?",
    actual_output=(
        "Here are several evidence-based strategies to improve sleep quality: "
        "1) Maintain a consistent sleep schedule, going to bed and waking up at the same time daily. "
        "2) Keep your bedroom cool (65-68F), dark, and quiet. "
        "3) Avoid screens for 30-60 minutes before bed, as blue light suppresses melatonin. "
        "4) Limit caffeine after 2pm. "
        "5) Exercise regularly, but not within 3 hours of bedtime. "
        "6) Try relaxation techniques like deep breathing or progressive muscle relaxation."
    ),
)

helpfulness_metric.measure(helpful_case)
print(f"Helpfulness Score (helpful response): {helpfulness_metric.score:.2f}")
print(f"Reason: {helpfulness_metric.reason}")

Output()

Helpfulness Score (helpful response): 0.99
Reason: The response directly addresses the user's query about improving sleep quality by providing a comprehensive list of evidence-based strategies. Each suggestion is practical and actionable, allowing the user to implement them easily. The output is relevant to the user's needs and covers all necessary aspects of sleep improvement, ensuring a complete solution.


In [ ]:
# Now test on a technically correct but unhelpful response
unhelpful_case = LLMTestCase(
    input="How can I improve my sleep quality?",
    actual_output=(
        "Sleep is a naturally recurring state of mind and body, characterized by altered consciousness "
        "and relatively inhibited sensory activity. It is regulated by circadian rhythms and sleep "
        "homeostasis. The stages of sleep include NREM stages 1-3 and REM sleep."
    ),
)

helpfulness_metric.measure(unhelpful_case)
print(f"Helpfulness Score (unhelpful response): {helpfulness_metric.score:.2f}")
print(f"Reason: {helpfulness_metric.reason}")
print(f"\nNotice: The unhelpful response is factually correct but does not address the user's actual need.")

G-Eval's power lies in its flexibility. You can define any evaluation criterion you want, and DeepEval will generate the appropriate Chain-of-Thought steps and use them to evaluate. This is the building block for domain-specific evaluation.

-

## Part 5: How RAGAS Uses LLM-as-a-Judge Under the Hood

RAGAS (Retrieval Augmented Generation Assessment) takes a different approach to LLM-as-a-Judge than DeepEval. Understanding these differences will help you choose the right framework for your use case.

### 5.1 RAGAS Faithfulness: Statement-Level NLI

RAGAS Faithfulness works similarly to DeepEval's but uses a Natural Language Inference (NLI) style verification:

1. **Extract statements** from the response
2. For each statement, determine if it can be **inferred from** the context (NLI: entailment/contradiction/neutral)
3. **Score** = number of supported statements / total statements

Let's simulate this manually.

In [ ]:
# We reuse the same test case from Part 4
print(f"Query:    {test_query}")
print(f"Context:  {len(test_context)} passages")
print(f"Response: {test_response[:80]}...")

# RAGAS-style NLI verification
nli_prompt_template = """Given the context below, determine if the statement can be inferred from the context.
Use Natural Language Inference:
- "entailment" if the context supports the statement
- "contradiction" if the context contradicts the statement
- "neutral" if the context neither supports nor contradicts

Context:
{context}

Statement: {statement}

Respond with JSON: {{"label": "entailment" or "contradiction" or "neutral", "reason": "<brief>"}}"""

nli_results = []
for claim in claims:  # reuse claims from Part 4
    prompt = nli_prompt_template.format(context=context_text, statement=claim)
    result = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=200,
    )
    raw = result.choices[0].message.content.strip()
    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        import re
        match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
        parsed = json.loads(match.group()) if match else {"label": "error", "reason": raw}
    nli_results.append({"claim": claim, **parsed})

print("\nRAGAS-style NLI Verification:")
print("=" * 70)
for r in nli_results:
    label = r['label'].upper()
    print(f"  [{label}] {r['claim']}")
    print(f"          {r['reason']}")

# Score (only "entailment" counts as supported)
supported = sum(1 for r in nli_results if r['label'] == 'entailment')
ragas_manual_faith = supported / len(nli_results) if nli_results else 0
print(f"\nRAGAS-style Manual Faithfulness: {ragas_manual_faith:.2f}")

Now let's run the same test through RAGAS.

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness

ragas_data = {
    "question": [test_query],
    "answer": [test_response],
    "contexts": [test_context],
    "ground_truth": ["The Eiffel Tower was built 1887-1889 for the World's Fair, designed by Gustave Eiffel's company, stands 330m tall."],
}

ragas_dataset = Dataset.from_dict(ragas_data)
ragas_results = evaluate(ragas_dataset, metrics=[faithfulness])

print(f"RAGAS Faithfulness Score: {ragas_results['faithfulness']:.2f}")
print(f"Manual Score:             {ragas_manual_faith:.2f}")
print(f"DeepEval Score:           {faithfulness_metric.score:.2f}")

### 5.2 RAGAS Answer Relevancy: Reverse-Engineering Questions

Here is where RAGAS gets creative. Instead of directly classifying statements as relevant/irrelevant, RAGAS:

1. **Generates N questions** from the response ("If this were an answer, what questions would it answer?")
2. **Computes embedding similarity** between each generated question and the original question
3. **Score** = average cosine similarity

The intuition: if the response is relevant, then questions reverse-engineered from it should be similar to the original question.

Let's simulate this approach manually.

In [ ]:
import numpy as np

# Step 1: Generate questions from the response
gen_q_prompt = f"""Given the following answer, generate 3 different questions that this answer could be responding to.
Return a JSON array of strings.

Answer: {relevancy_response}

Return ONLY a JSON array of 3 questions."""

gen_q_result = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": gen_q_prompt}],
    temperature=0,
    max_tokens=300,
)

gen_q_raw = gen_q_result.choices[0].message.content.strip()
if gen_q_raw.startswith("```"):
    gen_q_raw = gen_q_raw.split("```")[1]
    if gen_q_raw.startswith("json"):
        gen_q_raw = gen_q_raw[4:]
generated_questions = json.loads(gen_q_raw.strip())

print(f"Original question: {relevancy_query}\n")
print("Generated questions from the response:")
for i, q in enumerate(generated_questions, 1):
    print(f"  {i}. {q}")

In [ ]:
# Step 2: Compute embeddings for original and generated questions
all_questions = [relevancy_query] + generated_questions

embedding_response = client.embeddings.create(
    model="text-embedding-3-small",
    input=all_questions,
)

embeddings = [item.embedding for item in embedding_response.data]
original_emb = np.array(embeddings[0])
generated_embs = [np.array(emb) for emb in embeddings[1:]]

# Step 3: Compute cosine similarity
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

similarities = [cosine_similarity(original_emb, gen_emb) for gen_emb in generated_embs]
avg_similarity = np.mean(similarities)

print("Cosine Similarities with Original Question:")
for i, (q, sim) in enumerate(zip(generated_questions, similarities), 1):
    print(f"  {i}. [{sim:.4f}] {q}")

print(f"\nAverage Similarity (RAGAS-style Relevancy): {avg_similarity:.4f}")

Now let's verify with RAGAS.

In [ ]:
from ragas.metrics import answer_relevancy

ragas_rel_data = {
    "question": [relevancy_query],
    "answer": [relevancy_response],
    "contexts": [["Green tea has numerous health benefits including antioxidants and improved brain function."]],
    "ground_truth": ["Green tea is rich in antioxidants and may improve health."],
}

ragas_rel_dataset = Dataset.from_dict(ragas_rel_data)
ragas_rel_results = evaluate(ragas_rel_dataset, metrics=[answer_relevancy])

print(f"RAGAS Answer Relevancy:           {ragas_rel_results['answer_relevancy']:.4f}")
print(f"Manual Embedding Similarity:       {avg_similarity:.4f}")
print(f"DeepEval Answer Relevancy:         {ar_metric.score:.2f}")

### 5.3 Key Insight: DeepEval vs RAGAS Use DIFFERENT Approaches

Even though both frameworks measure "faithfulness" and "answer relevancy," they use fundamentally different LLM-as-a-Judge strategies:

| Metric | DeepEval Approach | RAGAS Approach |
|---|---|---|
| **Faithfulness** | Extract claims, then verify each claim against context | Extract statements, use NLI to check entailment from context |
| **Answer Relevancy** | Break into statements, classify each as relevant/irrelevant | Reverse-engineer questions from the answer, compute embedding similarity with original question |
| **Scoring** | Direct LLM classification | Hybrid (LLM generation + embedding similarity) |
| **Judge Model** | Configurable (default: GPT-4o) | Configurable (default: GPT-3.5-turbo) |

Neither approach is strictly "better" — they have different strengths and weaknesses. RAGAS's embedding-based relevancy is arguably more robust to judge model biases, while DeepEval's direct classification is more interpretable.

-

## Part 6: Customizing the Judge

### 6.1 Comparing Judge Models

The choice of judge model significantly affects evaluation scores. Stronger models are generally more reliable judges. Let's compare GPT-4o vs GPT-4o-mini on the same test cases.

In [ ]:
import pandas as pd

judge_models = ["gpt-4o-mini", "gpt-4o"]

test_cases = [
    {
        "query": "What is quantum computing?",
        "response": (
            "Quantum computing uses quantum bits (qubits) that can exist in superposition, "
            "allowing them to represent 0 and 1 simultaneously. This enables quantum computers "
            "to solve certain problems exponentially faster than classical computers."
        ),
        "label": "good",
    },
    {
        "query": "What is quantum computing?",
        "response": "Quantum computing is like regular computing but faster because it uses quantum.",
        "label": "vague",
    },
    {
        "query": "Explain the water cycle.",
        "response": (
            "The water cycle involves evaporation from bodies of water, condensation into clouds, "
            "precipitation as rain or snow, and collection in rivers, lakes, and oceans. "
            "Solar energy drives the entire process."
        ),
        "label": "good",
    },
    {
        "query": "Explain the water cycle.",
        "response": "Water goes up and then comes back down. This is the water cycle.",
        "label": "oversimplified",
    },
]

results_data = []

for tc in test_cases:
    row = {"query": tc["query"][:40] + "...", "label": tc["label"]}
    for model in judge_models:
        judgment = structured_judge(tc["query"], tc["response"], model=model)
        row[model] = judgment["score"]
    results_data.append(row)

df = pd.DataFrame(results_data)
print("Judge Model Comparison")
print("=" * 70)
print(df.to_string(index=False))
print(f"\nScore difference between models:")
for _, row in df.iterrows():
    diff = abs(row["gpt-4o"] - row["gpt-4o-mini"])
    print(f"  {row['label']}: |{row['gpt-4o']} - {row['gpt-4o-mini']}| = {diff}")

### 6.2 Custom Judge with DeepEval: DeepEvalBaseLLM

DeepEval allows you to plug in any LLM as the judge by subclassing `DeepEvalBaseLLM`. This is useful if you want to use a local model, a different provider, or a fine-tuned judge model.

In [ ]:
from deepeval.models import DeepEvalBaseLLM


class CustomOpenAIJudge(DeepEvalBaseLLM):
    """Custom judge using a specific OpenAI model with custom parameters."""

    def __init__(self, model_name: str = "gpt-4o-mini", temperature: float = 0):
        self.model_name = model_name
        self.temperature = temperature
        self._client = OpenAI()

    def load_model(self):
        return self.model_name

    def generate(self, prompt: str) -> str:
        response = self._client.chat.completions.create(
            model=self.model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=self.temperature,
            max_tokens=1000,
        )
        return response.choices[0].message.content

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self) -> str:
        return self.model_name


# Create a custom judge instance
custom_judge = CustomOpenAIJudge(model_name="gpt-4o-mini", temperature=0)

# Use it with a DeepEval metric
custom_metric = AnswerRelevancyMetric(
    threshold=0.7,
    model=custom_judge,
)

custom_test = LLMTestCase(
    input="What is the capital of France?",
    actual_output="Paris is the capital of France.",
)

custom_metric.measure(custom_test)
print(f"Custom judge score: {custom_metric.score:.2f}")
print(f"Reason: {custom_metric.reason}")
print(f"\nThis demonstrates how to plug any LLM into DeepEval's evaluation pipeline.")

### 6.3 Custom Rubrics for Domain-Specific Evaluation

G-Eval lets you design evaluation rubrics tailored to specific domains. Let's create three domain-specific rubrics and evaluate responses using each.

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

# Medical accuracy rubric
medical_metric = GEval(
    name="Medical Accuracy",
    criteria=(
        "Medical Accuracy - evaluate whether the response provides medically accurate information. "
        "Check for: correct medical terminology, accurate descriptions of conditions/treatments, "
        "appropriate disclaimers about seeking professional medical advice, and absence of "
        "dangerous misinformation."
    ),
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

medical_case = LLMTestCase(
    input="What are the symptoms of Type 2 diabetes?",
    actual_output=(
        "Common symptoms of Type 2 diabetes include increased thirst (polydipsia), frequent urination "
        "(polyuria), unexplained weight loss, fatigue, blurred vision, slow-healing sores, and frequent "
        "infections. However, many people with Type 2 diabetes may have no symptoms initially. If you "
        "experience these symptoms, please consult a healthcare provider for proper diagnosis."
    ),
)

medical_metric.measure(medical_case)
print(f"Medical Accuracy Score: {medical_metric.score:.2f}")
print(f"Reason: {medical_metric.reason}")

In [ ]:
# Customer support quality rubric
support_metric = GEval(
    name="Customer Support Quality",
    criteria=(
        "Customer Support Quality - evaluate the response as if it were from a customer support agent. "
        "Check for: empathetic tone, clear acknowledgment of the issue, actionable steps to resolve, "
        "professional language, and follow-up offer."
    ),
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

support_case = LLMTestCase(
    input="My order hasn't arrived and it's been 2 weeks. I'm very frustrated.",
    actual_output=(
        "I completely understand your frustration, and I sincerely apologize for the delay. "
        "Two weeks is certainly longer than expected. Let me look into this for you right away. "
        "Could you please provide your order number? In the meantime, I will escalate this to "
        "our shipping team. If we cannot locate your package, we will either reship the order "
        "or provide a full refund, whichever you prefer."
    ),
)

support_metric.measure(support_case)
print(f"Customer Support Score: {support_metric.score:.2f}")
print(f"Reason: {support_metric.reason}")

In [ ]:
# Code review quality rubric
code_review_metric = GEval(
    name="Code Review Quality",
    criteria=(
        "Code Review Quality - evaluate whether the code review feedback is constructive and actionable. "
        "Check for: identification of actual bugs or issues, clear explanation of why something is "
        "problematic, specific suggestions for improvement (not just criticism), appropriate tone, "
        "and prioritization of issues by severity."
    ),
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model="gpt-4o-mini",
    threshold=0.7,
)

code_review_case = LLMTestCase(
    input="Review this Python function: def get_user(id): return db.query('SELECT * FROM users WHERE id = ' + id)",
    actual_output=(
        "Critical issue: SQL injection vulnerability. The function concatenates user input directly "
        "into the SQL query. An attacker could pass something like '1; DROP TABLE users' as the id. "
        "Fix: Use parameterized queries instead: db.query('SELECT * FROM users WHERE id = %s', (id,)). "
        "Minor: Consider adding type hints (id: int) and a docstring. Also, 'SELECT *' is generally "
        "discouraged in production code -- select only the columns you need."
    ),
)

code_review_metric.measure(code_review_case)
print(f"Code Review Quality Score: {code_review_metric.score:.2f}")
print(f"Reason: {code_review_metric.reason}")

### 6.4 Temperature Effect on Judge Consistency

Temperature controls the randomness of the judge's output. Lower temperature means more deterministic (and generally more consistent) scores. Let's demonstrate how temperature affects score variance.

In [ ]:
def judge_with_temperature(query: str, response_text: str, temperature: float, n_runs: int = 5) -> list:
    """Run the same judge evaluation multiple times at a given temperature."""
    scores = []
    for _ in range(n_runs):
        result = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": f"Question: {query}\nResponse: {response_text}"},
            ],
            temperature=temperature,
            max_tokens=300,
        )
        raw = result.choices[0].message.content.strip()
        try:
            parsed = json.loads(raw)
            scores.append(parsed['score'])
        except (json.JSONDecodeError, KeyError):
            import re
            match = re.search(r'\{[^}]+\}', raw, re.DOTALL)
            if match:
                try:
                    scores.append(json.loads(match.group())['score'])
                except (json.JSONDecodeError, KeyError):
                    pass
    return scores


query = "Explain how vaccines work."
response_text = (
    "Vaccines work by training your immune system to recognize and fight specific pathogens. "
    "They contain weakened or inactivated forms of a virus or bacteria (or parts of them). "
    "When administered, the immune system produces antibodies and memory cells that provide "
    "future protection against the real pathogen."
)

temperatures = [0, 0.3, 0.7, 1.0]
temp_results = {}

for temp in temperatures:
    scores = judge_with_temperature(query, response_text, temp, n_runs=5)
    temp_results[temp] = scores

print("Temperature Effect on Judge Scores (5 runs each)")
print("=" * 60)
for temp, scores in temp_results.items():
    mean_score = np.mean(scores)
    std_score = np.std(scores)
    print(f"  Temp {temp}: scores={scores}, mean={mean_score:.2f}, std={std_score:.2f}")

print("\nKey insight: Higher temperature leads to more score variance.")
print("Always use temperature=0 for evaluation to ensure reproducibility.")

-

## Part 7: Best Practices and When NOT to Use LLM-as-a-Judge

### Best Practices

1. **Use the strongest available model as judge.** GPT-4o is a better judge than GPT-4o-mini, which is better than GPT-3.5. The judge should always be at least as capable as the model being evaluated.

2. **Design clear rubrics with examples.** Vague criteria like "rate quality" produce inconsistent results. Specific rubrics with score-level descriptions are essential.

3. **Run evaluations at temperature 0.** As demonstrated above, higher temperatures introduce unnecessary variance.

4. **Debias pairwise comparisons.** Always run both orderings and take consensus.

5. **Validate judge alignment with human scores.** Before deploying LLM-as-a-Judge, run a calibration study comparing LLM scores to human annotator scores on the same test set. Aim for high Spearman/Kendall correlation.

6. **Cache results to save cost.** The same (query, response, criteria) triple should always produce the same score (at temp=0). Cache aggressively.

7. **Use structured output.** JSON with mandatory reasoning fields prevents the judge from giving scores without justification.

8. **Be explicit about what NOT to evaluate.** Tell the judge "do not penalize brevity" or "ignore formatting" if those are not part of your criteria.

### When NOT to Use LLM-as-a-Judge

LLM-as-a-Judge is powerful but not always appropriate:

| Scenario | Better Alternative |
|---|---|
| **Exact match tasks** ("What is 2+2?") | Use exact string match or numeric comparison |
| **JSON/schema validation** ("Is the output valid JSON?") | Use `json.loads()` or a JSON schema validator |
| **Deterministic, reproducible scores needed** | Use rule-based metrics or embedding similarity |
| **Cost is prohibitive** | Use lighter metrics (BLEU, ROUGE, BERTScore) as proxies |
| **Judge is the same model being evaluated** | Self-enhancement bias will skew results |
| **Evaluating factual accuracy of cutting-edge knowledge** | The judge may not know the correct answer either |
| **High-stakes decisions (medical, legal)** | Use human experts — LLM judges can be confidently wrong |

### Cost Analysis

Let's estimate the cost of running LLM-as-a-Judge evaluations at scale.

In [ ]:
# Approximate token costs (as of early 2025, may change)
# GPT-4o: $2.50 per 1M input tokens, $10 per 1M output tokens
# GPT-4o-mini: $0.15 per 1M input tokens, $0.60 per 1M output tokens

pricing = {
    "gpt-4o": {"input": 2.50 / 1_000_000, "output": 10.00 / 1_000_000},
    "gpt-4o-mini": {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
}

# Typical token counts per evaluation
avg_input_tokens = 800   # prompt + rubric + query + response
avg_output_tokens = 150  # score + reason

# DeepEval Faithfulness requires ~2-3 LLM calls per test case
# RAGAS Answer Relevancy requires ~2 LLM calls + 1 embedding call
avg_calls_per_metric = 2.5

def estimate_cost(n_test_cases, n_metrics, model, calls_per_metric=2.5):
    total_calls = n_test_cases * n_metrics * calls_per_metric
    input_cost = total_calls * avg_input_tokens * pricing[model]["input"]
    output_cost = total_calls * avg_output_tokens * pricing[model]["output"]
    return input_cost + output_cost

print("Cost Estimation for LLM-as-a-Judge")
print("=" * 60)
print(f"Assumptions: ~{avg_input_tokens} input tokens, ~{avg_output_tokens} output tokens per call")
print(f"             ~{avg_calls_per_metric} LLM calls per metric per test case\n")

scenarios = [
    (50, 3, "Small eval: 50 test cases, 3 metrics"),
    (200, 5, "Medium eval: 200 test cases, 5 metrics"),
    (1000, 5, "Large eval: 1000 test cases, 5 metrics"),
    (5000, 8, "Enterprise eval: 5000 test cases, 8 metrics"),
]

for n_cases, n_metrics, desc in scenarios:
    cost_4o = estimate_cost(n_cases, n_metrics, "gpt-4o")
    cost_mini = estimate_cost(n_cases, n_metrics, "gpt-4o-mini")
    print(f"{desc}:")
    print(f"  GPT-4o:      ${cost_4o:.2f}")
    print(f"  GPT-4o-mini: ${cost_mini:.2f}")
    print(f"  Savings:     {(1 - cost_mini/cost_4o)*100:.0f}% with mini\n")

The cost difference between GPT-4o and GPT-4o-mini is substantial at scale. A common pattern is to use GPT-4o-mini for rapid iteration during development and GPT-4o for final evaluation runs.

-

## Part 8: Summary

### The Three Paradigms

| Paradigm | Use When | Pros | Cons |
|---|---|---|---|
| **Pointwise** | You need to score individual responses | Simple, supports criteria-specific scoring | Absolute scores are hard to calibrate |
| **Pairwise** | You need to compare two systems | More reliable relative judgments | Suffers from position bias, does not scale well with many candidates |
| **Reference-Based** | You have gold-standard answers | Grounded in a known-good answer | Requires reference answers, penalizes valid alternatives |

### Biases and Mitigations

| Bias | Mitigation |
|---|---|
| Position bias | Run both orderings, take consensus |
| Verbosity bias | Explicit rubric: "do not favor length" |
| Self-enhancement bias | Use a different/stronger model as judge |
| Format bias | Strip formatting or add "ignore formatting" to rubric |

### DeepEval vs RAGAS

Both frameworks build on LLM-as-a-Judge but take different approaches:

- **DeepEval**: Direct LLM classification for most metrics, supports G-Eval for custom criteria, verbose mode for debugging
- **RAGAS**: Hybrid approach combining LLM generation with embedding similarity, more novel metric designs

### What You Now Know

You have built LLM-as-a-Judge **from scratch**: pointwise, pairwise, and reference-based judges. You have seen the biases that affect judge reliability and how to mitigate them. You have peeked inside DeepEval and RAGAS to understand that every metric they compute is ultimately a structured LLM-as-a-Judge pipeline. And you know when to use this approach and when simpler alternatives are more appropriate.

**This is the engine behind everything in this workbook series.** Every metric in notebooks 03 through 10 is, at its core, an LLM-as-a-Judge call wrapped in a framework.

-

**Next:** Continue to [01_environment_setup.ipynb](01_environment_setup.ipynb) to install all dependencies and configure your environment for the rest of the series.